In [1]:
# Student Identity
LAST_NAME = "Quintela"
STUDENT_ID = "TUPM-26-3488"
FAVORITE_ARTIST = "The Weeknd"
SEED_NUM = 8

# Derive computational parameters from the fixed inputs
seed_digit = int(STUDENT_ID[-1])
id_checksum = sum(int(d) for d in STUDENT_ID if d.isdigit())
vector_dim = len(LAST_NAME)
artist_positions = [ord(c) - 64 for c in FAVORITE_ARTIST if c.isalpha()]

# Encapsulate state into a Dictionary
sys_config = {
    "operator": LAST_NAME,
    "artist": FAVORITE_ARTIST,
    "authid": seed_digit,
    "baseseed": SEED_NUM,
    "checksum": id_checksum,
    "vectordim": vector_dim,
    "positions": artist_positions,
    "status": "INITIALIZED"
}

print("=== SYSTEM CONFIGURATION ===")
for key, value in sys_config.items():
    print(f"{key.upper()}: {value}")


=== SYSTEM CONFIGURATION ===
OPERATOR: Quintela
ARTIST: The Weeknd
AUTHID: 8
BASESEED: 8
CHECKSUM: 31
VECTORDIM: 8
POSITIONS: [20, 40, 37, 23, 37, 37, 43, 46, 36]
STATUS: INITIALIZED


In [2]:
sys_config = {
    "operator": LAST_NAME,
    "artist": FAVORITE_ARTIST,
    "baseseed": SEED_NUM,
    "checksum": id_checksum,
    "vectordim": vector_dim,
    "positions": artist_positions,
    "status": "INITIALIZED"
}
print("=== SYSTEM CONFIGURATION ===")
for key, value in sys_config.items():
    print(f"{key.upper()}: {value}")


=== SYSTEM CONFIGURATION ===
OPERATOR: Quintela
ARTIST: The Weeknd
BASESEED: 8
CHECKSUM: 31
VECTORDIM: 8
POSITIONS: [20, 40, 37, 23, 37, 37, 43, 46, 36]
STATUS: INITIALIZED


In [3]:
base = sys_config["baseseed"]
positions = sys_config["positions"]

customers = [f"{sys_config['operator']}_C{i}" for i in range(1, 6)]
products = ["NOTEBOOK", "PEN", "USB", "HEADPHONES", "CALCULATOR"]
prices = {"NOTEBOOK": 59.00, "PEN": 15.50, "USB": 320.00,
          "HEADPHONES": 899.00, "CALCULATOR": 450.00}

transactions = []
for row, cust in enumerate(customers):
    for col, prod in enumerate(products):
        if (row + col) % 3 == 0:          # not every customer buys everything
            qty = 1 + (base * (row + 1) + positions[(row + col) % len(positions)]) % 4
            transactions.append((cust, prod, qty, prices[prod]))

# At least one repeated customer-product transaction
transactions.append((customers[0], products[0], 2, prices[products[0]]))

print("=== GENERATED TRANSACTION RECORDS ===")
for t in transactions:
    print(t)
print(f"\nTotal records: {len(transactions)}")


=== GENERATED TRANSACTION RECORDS ===
('Quintela_C1', 'NOTEBOOK', 1, 59.0)
('Quintela_C1', 'HEADPHONES', 4, 899.0)
('Quintela_C2', 'USB', 4, 320.0)
('Quintela_C3', 'PEN', 4, 15.5)
('Quintela_C3', 'CALCULATOR', 4, 450.0)
('Quintela_C4', 'NOTEBOOK', 4, 59.0)
('Quintela_C4', 'HEADPHONES', 4, 899.0)
('Quintela_C5', 'USB', 4, 320.0)
('Quintela_C1', 'NOTEBOOK', 2, 59.0)

Total records: 9


In [4]:
organized = {}
for cust, prod, qty, price in transactions:
    value = round(qty * price, 2)
    organized.setdefault(cust, []).append((prod, qty, price, value))

total_value = round(sum(v for cust in organized for *_, v in organized[cust]), 2)
overall = sorted({prod for _, prod, _, _ in transactions})
distinct_customers = sorted(organized)

print("=== ORGANIZED TRANSACTION DATA ===")
for cust in distinct_customers:
    print(f"\nCustomer: {cust}")
    for prod, qty, price, value in organized[cust]:
        print(f"  {prod:<12} qty={qty}  price={price:<7} value={value}")
print(f"\nOverall transaction value: {total_value}")
print(f"Distinct products : {len(overall)} -> {overall}")
print(f"Distinct customers: {len(distinct_customers)}")


=== ORGANIZED TRANSACTION DATA ===

Customer: Quintela_C1
  NOTEBOOK     qty=1  price=59.0    value=59.0
  HEADPHONES   qty=4  price=899.0   value=3596.0
  NOTEBOOK     qty=2  price=59.0    value=118.0

Customer: Quintela_C2
  USB          qty=4  price=320.0   value=1280.0

Customer: Quintela_C3
  PEN          qty=4  price=15.5    value=62.0
  CALCULATOR   qty=4  price=450.0   value=1800.0

Customer: Quintela_C4
  NOTEBOOK     qty=4  price=59.0    value=236.0
  HEADPHONES   qty=4  price=899.0   value=3596.0

Customer: Quintela_C5
  USB          qty=4  price=320.0   value=1280.0

Overall transaction value: 12027.0
Distinct products : 5 -> ['CALCULATOR', 'HEADPHONES', 'NOTEBOOK', 'PEN', 'USB']
Distinct customers: 5


In [5]:
pair_list = [(cust, prod) for cust, prod, _, _ in transactions]
pair_set = set(pair_list)
repeated = [pair for pair in pair_set if pair_list.count(pair) > 1]

print("=== REPEATED / DISTINCT TRANSACTIONS ===")
print(f"Total records     : {len(pair_list)}")
print(f"Distinct purchases: {len(pair_set)}")
print(f"Repeated purchases: {repeated}")


=== REPEATED / DISTINCT TRANSACTIONS ===
Total records     : 9
Distinct purchases: 8
Repeated purchases: [('Quintela_C1', 'NOTEBOOK')]


In [6]:
customer_summary = {}
product_summary = {}

for cust, prod, qty, price in transactions:
    value = round(qty * price, 2)
    customer_summary[cust] = round(customer_summary.get(cust, 0) + value, 2)

    prev_qty, prev_val = product_summary.get(prod, (0, 0))
    product_summary[prod] = (prev_qty + qty, round(prev_val + value, 2))

print("=== CUSTOMER SUMMARIES ===")
for cust in sorted(customer_summary, key=customer_summary.get, reverse=True):
    print(f"{cust:<20} total = {customer_summary[cust]}")

print("\n=== PRODUCT SUMMARIES ===")
for prod in sorted(product_summary, key=lambda p: product_summary[p][1], reverse=True):
    qty, val = product_summary[prod]
    print(f"{prod:<12} sold={qty:<4} revenue={val}")


=== CUSTOMER SUMMARIES ===
Quintela_C4          total = 3832.0
Quintela_C1          total = 3773.0
Quintela_C3          total = 1862.0
Quintela_C2          total = 1280.0
Quintela_C5          total = 1280.0

=== PRODUCT SUMMARIES ===
HEADPHONES   sold=8    revenue=7192.0
USB          sold=8    revenue=2560.0
CALCULATOR   sold=4    revenue=1800.0
NOTEBOOK     sold=7    revenue=413.0
PEN          sold=4    revenue=62.0


In [7]:
top_customers = sorted(customer_summary.items(), key=lambda i: i[1], reverse=True)
top_products = sorted(product_summary.items(), key=lambda i: i[1][1], reverse=True)

print("=== ORDERED RESULTS ===")
print("Top customers by spend:")
for rank, (cust, total) in enumerate(top_customers, start=1):
    print(f"  {rank}. {cust:<20} {total}")
print("Top products by revenue:")
for rank, (prod, (qty, val)) in enumerate(top_products, start=1):
    print(f"  {rank}. {prod:<12} {val}")

print("\n=== FINAL CONSOLIDATED TRANSACTION REPORT ===")
print(f"Operator          : {sys_config['operator']}")
print(f"Records           : {len(transactions)}")
print(f"Customers         : {len(distinct_customers)}")
print(f"Products          : {len(overall)}")
print(f"Repeated purchases: {len(repeated)}")
print(f"Overall value     : {total_value}")
print(f"Top customer      : {top_customers[0][0]} ({top_customers[0][1]})")
print(f"Top product       : {top_products[0][0]} ({top_products[0][1][1]})")


=== ORDERED RESULTS ===
Top customers by spend:
  1. Quintela_C4          3832.0
  2. Quintela_C1          3773.0
  3. Quintela_C3          1862.0
  4. Quintela_C2          1280.0
  5. Quintela_C5          1280.0
Top products by revenue:
  1. HEADPHONES   7192.0
  2. USB          2560.0
  3. CALCULATOR   1800.0
  4. NOTEBOOK     413.0
  5. PEN          62.0

=== FINAL CONSOLIDATED TRANSACTION REPORT ===
Operator          : Quintela
Records           : 9
Customers         : 5
Products          : 5
Repeated purchases: 1
Overall value     : 12027.0
Top customer      : Quintela_C4 (3832.0)
Top product       : HEADPHONES (7192.0)
